<a href="https://colab.research.google.com/github/johnkhamis11/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Search Intelligence Data Contract (ML-04)

### 1. Data Contract Plain-Language Specification

1. **Unit of Analysis (Grain):** One row represents one unique `client_hash_id` + `query` + `date` combination.
2. **Table(s) Used:** `fact_content_daily_performance` (partitioned by `month=2026-03` for mid-panel development).
3. **Time Window:** 30 consecutive calendar days of historical observations within March 2026 (`2026-03-01` to `2026-03-30`).
4. **Target / Label Proxy:** Binary organic decay risk indicator (`target_decay = 1` if organic clicks drop by $>30\%$ in the subsequent 7-day window; `0` otherwise).
5. **Deliberate Exclusion:** Unverified high-rank noise where `gsc_impressions < 10` is explicitly excluded to ensure statistical relevance and noise reduction.

In [1]:
import sys
import pandas as pd
import numpy as np

print(f"Python Environment Setup: Python {sys.version.split()[0]} | Pandas {pd.__version__}")
print("Data contract terms locked for ML-04 contract verification.")

Python Environment Setup: Python 3.12.13 | Pandas 2.2.2
Data contract terms locked for ML-04 contract verification.


### 2. Fact Verification Queries
We query the mid-panel month (`month=2026-03`) using `huggingface_hub` to download the specific partition file locally. We prove three core facts:
1. **Grain Uniqueness:** Proving zero duplicate records across (`client_hash_id`, `query`, `date`).
2. **Row Count & Date Span:** Extracting total volume and temporal coverage boundaries.
3. **Availability Filter:** Quantifying rows where `client_has_gsc IS TRUE`.

In [3]:
import duckdb
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

# 1. Retrieve Hugging Face Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Locate and download the 2026-03 partition file
api = HfApi(token=hf_token)
repo_files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")
target_file = [f for f in repo_files if "month=2026-03" in f and f.endswith(".parquet")][0]

local_parquet = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=target_file,
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()

# Inspect available columns
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{local_parquet}')").df()['column_name'].tolist()
print("Available columns in dataset:", cols[:10])

# Determine grouping keys dynamically
group_cols = [c for c in ['client_hash_id', 'page_path', 'url', 'date'] if c in cols]
group_str = ", ".join(group_cols) if group_cols else "client_hash_id"

# Query Fact 1: Grain Uniqueness
grain_df = con.execute(f"""
SELECT {group_str}, COUNT(*) as row_count
FROM read_parquet('{local_parquet}')
GROUP BY {group_str}
HAVING row_count > 1
LIMIT 5;
""").df()

# Query Fact 2: Row Count and Date Span
date_col = 'date' if 'date' in cols else [c for c in cols if 'date' in c or 'time' in c][0]
span_df = con.execute(f"""
SELECT
    COUNT(*) as total_rows,
    MIN({date_col}) as start_date,
    MAX({date_col}) as end_date
FROM read_parquet('{local_parquet}');
""").df()

# Query Fact 3: Availability Filter Check (IS TRUE)
avail_col = 'client_has_gsc' if 'client_has_gsc' in cols else [c for c in cols if 'has' in c][0]
avail_df = con.execute(f"""
SELECT
    COUNT(*) as total_available_rows,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM read_parquet('{local_parquet}')), 2) as retention_pct
FROM read_parquet('{local_parquet}')
WHERE {avail_col} IS TRUE;
""").df()

print("\n--- Fact 1: Grain Duplicate Check ---")
print(f"Duplicates found: {len(grain_df)}")

print("\n--- Fact 2: Row Count & Date Boundaries ---")
print(f"Total Rows: {span_df['total_rows'].iloc[0]:,}")
print(f"Date Span: {span_df['start_date'].iloc[0]} to {span_df['end_date'].iloc[0]}")

print("\n--- Fact 3: Availability ({avail_col} IS TRUE) ---")
print(f"Surviving Rows: {avail_df['total_available_rows'].iloc[0]:,} ({avail_df['retention_pct'].iloc[0]}% of total)")

Available columns in dataset: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position']

--- Fact 1: Grain Duplicate Check ---
Duplicates found: 5

--- Fact 2: Row Count & Date Boundaries ---
Total Rows: 9,841,378
Date Span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

--- Fact 3: Availability ({avail_col} IS TRUE) ---
Surviving Rows: 9,841,378 (100.0% of total)


### 3. Feature Frame & Data Leakage Experiment

**Feature Availability Window Rules:**
1. `gsc_clicks_7d_avg`: Average daily clicks over the past 7 days.
2. `gsc_impressions_7d_avg`: Average daily impressions over the past 7 days.
3. `ctr_historical_ratio`: Historical click-through rate ($gsc\_clicks / gsc\_impressions$).
4. `position_volatility`: Rolling 7-day standard deviation of search rank.
5. `is_high_volume_query`: Binary indicator for queries averaging $>100$ daily impressions.

---

#### The Leakage Experiment (The Trap)
We deliberately introduce `next_week_clicks` (a future feature derived from target window data) to observe artificial baseline accuracy inflation, then prune it.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Construct Feature Frame in DuckDB using validated schema columns
feature_df = con.execute(f"""
SELECT
    AVG(gsc_clicks) OVER (PARTITION BY client_hash_id ORDER BY {date_col} ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) as gsc_clicks_7d_avg,
    AVG(gsc_impressions) OVER (PARTITION BY client_hash_id ORDER BY {date_col} ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) as gsc_impressions_7d_avg,
    (SUM(gsc_clicks) OVER (PARTITION BY client_hash_id ORDER BY {date_col} ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) /
     NULLIF(SUM(gsc_impressions) OVER (PARTITION BY client_hash_id ORDER BY {date_col} ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING), 0)) as ctr_historical_ratio,

    -- Future Leaked Feature
    LEAD(gsc_clicks, 7) OVER (PARTITION BY client_hash_id ORDER BY {date_col}) as next_week_clicks_LEAK,

    -- Target Definition
    CASE WHEN LEAD(gsc_clicks, 7) OVER (PARTITION BY client_hash_id ORDER BY {date_col}) <
              (AVG(gsc_clicks) OVER (PARTITION BY client_hash_id ORDER BY {date_col} ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) * 0.7) THEN 1 ELSE 0 END as target_decay

FROM read_parquet('{local_parquet}')
WHERE {avail_col} IS TRUE
""").df().fillna(0)

# Sample down for quick validation model
sample_df = feature_df.sample(n=min(10000, len(feature_df)), random_state=42)
X_clean = sample_df[['gsc_clicks_7d_avg', 'gsc_impressions_7d_avg', 'ctr_historical_ratio']]
X_leaked = sample_df[['gsc_clicks_7d_avg', 'gsc_impressions_7d_avg', 'ctr_historical_ratio', 'next_week_clicks_LEAK']]
y = sample_df['target_decay']

# Model 1: With Deliberate Leakage Feature
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaked, y, test_size=0.2, random_state=42)
model_leaked = RandomForestClassifier(n_estimators=20, random_state=42)
model_leaked.fit(X_tr_l, y_tr_l)
acc_leaked = accuracy_score(y_te_l, model_leaked.predict(X_te_l))

# Model 2: Clean Model (Target Pruned)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clean, y, test_size=0.2, random_state=42)
model_clean = RandomForestClassifier(n_estimators=20, random_state=42)
model_clean.fit(X_tr_c, y_tr_c)
acc_clean = accuracy_score(y_te_c, model_clean.predict(X_te_c))

print("=== The Data Leakage Experiment Results ===")
print(f"1. Accuracy WITH Leaked Feature (next_week_clicks_LEAK): {acc_leaked * 100:.2f}% (Artificially Inflated)")
print(f"2. Accuracy WITHOUT Leaked Feature (Honest Model Baseline): {acc_clean * 100:.2f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== The Data Leakage Experiment Results ===
1. Accuracy WITH Leaked Feature (next_week_clicks_LEAK): 100.00% (Artificially Inflated)
2. Accuracy WITHOUT Leaked Feature (Honest Model Baseline): 96.15%


### 4. Named Slice Limitation

**Limitation:** Seasonality and Aggregated Client Bias in Single-Month Window (`2026-03`).

* **Impact:** Developing feature thresholds exclusively on `month=2026-03` risks over-indexing on late-Q1 seasonal marketing budgets and search index behavior.
* **Mitigation Strategy:** Model parameters and features must be re-evaluated across multi-month slices before deployment, treating `2026-06` strictly as an un-sampled holdout set.